# Lakelink Fiber PRI: v4 candidate writers (@ml-engineer)

**Task:** P37-ML-V4-WRITERS-CHECKIN. This is the versioned writer for the two ML shadow objects that P34-ML-V4-RESCORE-DRYRUN built in an ad-hoc session on 2026-10-25 around 21:00 UTC:

| Target | Grain | Built from |
| --- | --- | --- |
| `span_risk_predictions_v4_candidate` | 1 row per span (10,092) | UC `span_risk_classifier@champion` (v7). 7 hazard features come from in-radius pairs in `hazard_span_intersection_v4_candidate`. The other 28 come from `span_risk_inference_log`. Includes a v3@22 rule control and the production class. |
| `resilience_score_snapshot_v4_candidate` | 1 row per service (8,000) | V2 formula (HL20, SLA boost, capped at 100) with active hazards from the v4 HSI at the v3 score_time. Diversity and telemetry are carried over from FROZEN v3 snapshot v22. |

**Freeze rules (DEC-PRI-V4-SHADOW-BUILD):** writes go only to `*_v4_candidate` targets. FROZEN v3 objects are only read, through the fail-closed fingerprint gate `v3src(table, pin)` (P65 / DEC-PRI-V4-PIN-FINGERPRINT-GATES). The gate uses the current table when (row count, v1, v2, column count) equals the `PRI-DEMO-FREEZE-v3` baseline in `freeze_pin_fingerprint_gate`. Otherwise it tries `VERSION AS OF pin`, and if neither matches it raises an error. No ungated time travel remains. `WRITE_ENABLED` defaults to `false`, so a normal run builds the frames and checks parity against the live candidate tables without writing anything. `INSERT OVERWRITE` keeps the existing table and column comments.

**Parity proof (S38, 2026-10-26, read-only):** predictions class 10,092/10,092 (v4) and 10,092/10,092 (v3rule control). Hazard counts 10,092/10,092. Snapshot active_hazards 8,000/8,000, hazard_exposure 8,000/8,000, overall_score 7,999/8,000 (one float tie off by 0.01), risk_band 8,000/8,000. KPIs: avg 56.77, critical 1,087, revenue at risk $54.44M.

In [0]:
# BUG-QA-093: span_risk_classifier@champion (v7 pyfunc) needs xgboost==3.4.1, which is not on base
# serverless. Install it first so a clean Run-All works. No logic changes.
%pip install -q xgboost==3.4.1
dbutils.library.restartPython()

In [0]:
import mlflow
import pandas as pd
from pyspark.sql import functions as F

dbutils.widgets.dropdown("WRITE_ENABLED", "false", ["false", "true"], "Write to *_v4_candidate")
dbutils.widgets.text("LIFT_TASK", "", "Approved task code (optional)")
WRITE_ENABLED = dbutils.widgets.get("WRITE_ENABLED") == "true"
LIFT_TASK = dbutils.widgets.get("LIFT_TASK") or None

SCHEMA = "cdm_tmforum.provable_resilience_intelligence"
MODEL_NAME = f"{SCHEMA}.span_risk_classifier"
MODEL_ALIAS = "champion"

# Inputs (read only)
HSI_V4 = f"{SCHEMA}.hazard_span_intersection_v4_candidate"
HSI_V3_PIN = 22
# P65-ML-SNAPSHOT-PIN-FP-GATE / BUG-QA-096 / DEC-PRI-V4-PIN-FINGERPRINT-GATES: this pin is an
# INFORMATIONAL label. The v3 snapshot is read through v3src() below, never through an ungated
# VERSION AS OF (snapshot@22 leaves the 7-day retention window at about 2026-10-31 13:08Z).
SNAPSHOT_V3_PIN = 22
FREEZE_GATE = f"{SCHEMA}.freeze_pin_fingerprint_gate"
FREEZE_MANIFEST_V3 = "PRI-DEMO-FREEZE-v3"
INFERENCE_LOG = f"{SCHEMA}.span_risk_inference_log"
SERVICE_INVENTORY = f"{SCHEMA}.service_inventory"

# Outputs (shadow only)
OUT_PRED = f"{SCHEMA}.span_risk_predictions_v4_candidate"
OUT_SNAP = f"{SCHEMA}.resilience_score_snapshot_v4_candidate"

# Held at the v3 snapshot v22 score_time so the deltas are like-for-like
SCORE_TIME = "2026-10-24 13:08:47"
# P36 / DEC-PRI-V4-RESCORE-HAZARD-SOURCE (Option A): v7 hazard features come from proximity
# semantics (P47), NOT HSI in-radius counts (out of distribution for v7).
PROX_V4 = f"{SCHEMA}.hazard_span_proximity_v4_candidate"
PROX_V3 = f"{SCHEMA}.hazard_span_proximity"
SPAN_V4 = f"{SCHEMA}.physical_resource_span_v4_candidate"
FEATURE_SOURCE = "hazard_span_proximity_v4_candidate@v1 (<=10 km proximity, v3 rule) + span_risk_inference_log (28 non-hazard features; BBS-* inherit parent BB-* route)"

FEATURES = [
    "fiber_count", "wavelength_count", "length_km", "span_age_years",
    "total_alarms", "distinct_alarm_types", "critical_alarm_count", "major_alarm_count",
    "severe_alarm_count", "osnr_degrade_count", "ber_high_count", "fiber_cut_count", "los_count",
    "avg_alarm_duration_hrs", "unack_alarm_count", "days_since_last_critical",
    "srlg_count", "srlg_type_count", "srlg_critical_count", "srlg_severe_count",
    "service_count", "spans_in_conduit",
    "wildfire_count", "flood_count", "dig_ticket_count", "weather_count", "earthquake_count",
    "hazard_total_count", "hazard_critical_count",
    "avg_optical_power_dbm", "avg_osnr_db", "avg_ber", "avg_pmd_ps", "stddev_osnr", "stddev_ber",
]
HAZARD_FEATURES = ["wildfire_count", "flood_count", "dig_ticket_count", "weather_count",
                   "earthquake_count", "hazard_total_count", "hazard_critical_count"]
assert len(FEATURES) == 35


def fingerprint(df):
    """(row count, v1 null-skip, v2 null-token, column count). Same definitions as the
    freeze_pin_fingerprint_gate columns fp_row_count / fp_sha2_v1_null_skip / fp_sha2_v2_null_token / fp_column_count."""
    cols = df.columns
    h1 = F.sha2(F.concat_ws("|", *[F.col(f"`{c}`").cast("string") for c in cols]), 256)
    h2 = F.sha2(F.concat_ws("|", *[F.coalesce(F.col(f"`{c}`").cast("string"), F.lit("<NULL>")) for c in cols]), 256)
    r = df.select(F.count("*").alias("n"),
                  F.sum(F.conv(F.substring(h1, 1, 15), 16, 10).cast("decimal(38,0)")).alias("v1"),
                  F.sum(F.conv(F.substring(h2, 1, 15), 16, 10).cast("decimal(38,0)")).alias("v2")).first()
    return (int(r.n), str(r.v1 or 0), str(r.v2 or 0), len(cols))


_V3SRC = {}


def v3src(table: str, pin: int) -> str:
    """Return a temp view that holds table@pin, gated on the PRI-DEMO-FREEZE-v3 baseline in
    freeze_pin_fingerprint_gate (DE S61 v3src pattern). Fails closed.
    Tries 1) the current table, 2) VERSION AS OF pin while it is still within retention. The first
    source whose (row count, v1, v2, column count) equals the baseline wins. Otherwise AssertionError."""
    key = (table, int(pin))
    if key in _V3SRC:
        return _V3SRC[key]
    base = spark.sql(f"""
        SELECT fp_row_count, fp_sha2_v1_null_skip, fp_sha2_v2_null_token, fp_column_count,
               manifest_delta_version, gate_status
        FROM {FREEZE_GATE}
        WHERE manifest_version = '{FREEZE_MANIFEST_V3}' AND object_name = '{table}'""").collect()
    assert len(base) == 1, f"FP gate: expected 1 {FREEZE_MANIFEST_V3} baseline row for {table}, got {len(base)}; refusing an ungated read"
    b = base[0]
    assert b.gate_status == "PASS", f"FP gate: baseline for {table} is {b.gate_status}, not PASS"
    assert int(b.manifest_delta_version) == int(pin), f"FP gate: baseline is for {table}@{b.manifest_delta_version}, not @{pin}"
    want = (int(b.fp_row_count), str(b.fp_sha2_v1_null_skip), str(b.fp_sha2_v2_null_token), int(b.fp_column_count))
    full = f"{SCHEMA}.{table}"
    tried = []
    for mode, q in (("current", f"SELECT * FROM {full}"),
                    ("time_travel", f"SELECT * FROM {full} VERSION AS OF {int(pin)}")):
        try:
            got = fingerprint(spark.sql(q))
        except Exception as e:  # e.g. DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION
            tried.append((mode, type(e).__name__))
            continue
        if got == want:
            view = f"fp_{table}_{int(pin)}"
            spark.sql(f"CREATE OR REPLACE TEMP VIEW {view} AS {q}")
            print(f"PASS FP gate {table}@{pin} via {mode}: {got}")
            _V3SRC[key] = view
            return view
        tried.append((mode, got))
    raise AssertionError(f"FP gate failed closed for {table}@{pin}: want {want}, tried {tried}")

print(f"WRITE_ENABLED={WRITE_ENABLED} | outputs: {OUT_PRED}, {OUT_SNAP}")

In [0]:
# P48-ML-V4-WRITERS-CHECKIN: versioned writers for the two ML upstream hazard objects that were
# built inline in agent sessions (P47 and P44). Dry run by default: recompute and check parity
# against the live candidate objects. Writes ONLY when WRITE_ENABLED=true AND LIFT_TASK is one of
# UPSTREAM_TASKS, so a P36 rescore run (LIFT_TASK=P36-...) never rebuilds the upstream inputs.
# Targets are *_v4_candidate only; FROZEN v3 objects are read only.
OUT_PROX = f"{SCHEMA}.hazard_span_proximity_v4_candidate"
OUT_HSI_FEAT = f"{SCHEMA}.span_hazard_features_v4_candidate"
HAZARD_EVENT = f"{SCHEMA}.geographic_hazard_event"
UPSTREAM_TASKS = {"P48-ML-V4-WRITERS-CHECKIN", "P47-ML-HAZARD-PROXIMITY-V4-CANDIDATE", "P44-ML-HSI-BBS-GRAIN-FEATURES"}
UPSTREAM_WRITE = WRITE_ENABLED and LIFT_TASK in UPSTREAM_TASKS

# --- (1) hazard_span_proximity_v4_candidate (P47, DEC-PRI-V4-RESCORE-HAZARD-SOURCE Option A) ---
# Exact v3 hazard_span_proximity rule re-applied to the v4 span grain (SPAN-* + BBS-*):
# v3 hazard vintage (created 2026-10-19), degree box |dlat|,|dlon| < 0.12 (load-bearing),
# haversine R=6371.0 from epicenter to span midpoint, keep <= 10 km (radius_km NOT used).
PROX_V4_SELECT = f"""
WITH hz AS (
  SELECT hazard_id, hazard_type, severity AS hazard_severity, event_start, event_end, radius_km, latitude, longitude
  FROM {HAZARD_EVENT}
  WHERE date(created_at) = DATE'2026-10-19'
),
sp AS (
  SELECT span_id, parent_route_id, midpoint_lat, midpoint_lon
  FROM {SPAN_V4}
  WHERE midpoint_lat IS NOT NULL AND midpoint_lon IS NOT NULL
),
pairs AS (
  SELECT hz.hazard_id, hz.hazard_type, hz.hazard_severity, hz.event_start, hz.event_end, hz.radius_km,
         sp.span_id, sp.parent_route_id,
         6371.0 * 2 * asin(sqrt(
           pow(sin(radians(sp.midpoint_lat - hz.latitude) / 2), 2) +
           cos(radians(hz.latitude)) * cos(radians(sp.midpoint_lat)) *
           pow(sin(radians(sp.midpoint_lon - hz.longitude) / 2), 2))) AS distance_km
  FROM hz JOIN sp
    ON abs(sp.midpoint_lat - hz.latitude) < 0.12
   AND abs(sp.midpoint_lon - hz.longitude) < 0.12
)
SELECT hazard_id, hazard_type, hazard_severity, event_start, event_end, radius_km,
       span_id, parent_route_id, distance_km, 'DERIVED' AS origin
FROM pairs WHERE distance_km <= 10
"""

PROX_V4_DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.hazard_span_proximity_v4_candidate (
  hazard_id STRING COMMENT 'FK to geographic_hazard_event.hazard_id',
  hazard_type STRING COMMENT 'Hazard type: dig_ticket | wildfire | flood | severe_weather | earthquake',
  hazard_severity STRING COMMENT 'Hazard severity: critical | high | medium | low',
  event_start TIMESTAMP COMMENT 'When the hazard event began or was reported (current value from geographic_hazard_event)',
  event_end TIMESTAMP COMMENT 'When the hazard event ended (NULL if ongoing); current value from geographic_hazard_event, so it can differ from v3 where the source closed an event later. Not a model feature input.',
  radius_km DOUBLE COMMENT 'Original hazard impact radius in km (informational; the proximity rule uses a fixed 10 km threshold, not this radius)',
  span_id STRING COMMENT 'FK to physical_resource_span_v4_candidate.span_id. SPAN-* = unchanged access/metro spans; BBS-* = planar backbone route segments that replace the retired BB-* spans. No BB-* ids.',
  parent_route_id STRING COMMENT 'For BBS-* segments, the backbone route the segment belongs to (from physical_resource_span_v4_candidate.parent_route_id). NULL for SPAN-* spans.',
  distance_km DOUBLE COMMENT 'Haversine distance in km (R = 6371.0) from hazard epicenter (latitude/longitude) to span or segment midpoint. Rows kept when distance_km <= 10.',
  origin STRING COMMENT 'Data provenance: SYNTHETIC, INGESTED, DERIVED, INFERENCE, REFERENCE or MONITORING (PRI origin-tracking standard P14-DP). Always DERIVED here.'
) USING delta
COMMENT 'v4 CANDIDATE (shadow, not FROZEN v3). Lakelink Fiber Provable Resilience Intelligence. Hazard-to-span proximity with the exact v3 hazard_span_proximity semantics, re-applied to physical_resource_span_v4_candidate (SPAN-* + BBS-* segments). Feeds the champion span-risk v7 hazard features for the P36 v4 shadow rescore per DEC-PRI-V4-RESCORE-HAZARD-SOURCE. Built by @ml-engineer task P47; writer checked in by P48.'
"""

# --- (2) span_hazard_features_v4_candidate (P44, Option B v8 challenger input; NOT fed to v7) ---
HSI_FEAT_VIEW_DDL = f"""
CREATE OR REPLACE VIEW {SCHEMA}.span_hazard_features_v4_candidate (
  span_id COMMENT 'Unique span identifier. SPAN-* = real 1-25 km conduit segments (unchanged from v3 pin span@20). BBS-<route>-NNN = v4 backbone segments split from v3 BB-* routes (BUG-DP-068 / V4 B1).',
  route_key COMMENT 'Aggregation key for hazard features: parent_route_id for BBS-* backbone segments, otherwise span_id for SPAN-* spans. Never NULL.',
  feature_grain COMMENT 'Grain of the hazard counts on this row: parent_route = counts over the whole parent backbone route, broadcast to every BBS-* segment (do not SUM across segments); span = counts for this SPAN-* span only.',
  hazard_total_count COMMENT 'Count of DISTINCT hazard_id in hazard_span_intersection_v4_candidate with is_within_radius = TRUE for route_key (all types and severities). 0 when none. DOUBLE.',
  hazard_critical_count COMMENT 'Count of DISTINCT in-radius hazard_id for route_key with hazard_severity critical. 0 when none. DOUBLE.',
  wildfire_count COMMENT 'Count of DISTINCT in-radius hazard_id for route_key with hazard_type wildfire. 0 when none. DOUBLE.',
  flood_count COMMENT 'Count of DISTINCT in-radius hazard_id for route_key with hazard_type flood. 0 when none. DOUBLE.',
  weather_count COMMENT 'Count of DISTINCT in-radius hazard_id for route_key with hazard_type severe_weather. 0 when none. DOUBLE.',
  earthquake_count COMMENT 'Count of DISTINCT in-radius hazard_id for route_key with hazard_type earthquake. 0 when none. DOUBLE.',
  dig_ticket_count COMMENT 'Count of DISTINCT in-radius hazard_id for route_key with hazard_type dig_ticket (811 excavation tickets). 0 when none. DOUBLE.',
  in_radius_row_count COMMENT 'Raw count of in-radius hazard_span_intersection_v4_candidate rows for route_key before hazard_id de-duplication. Diagnostic only; in_radius_row_count >= hazard_total_count.'
)
COMMENT 'Lakelink Fiber span-risk hazard-exposure features, grain-safe for v4 (P44-ML-HSI-BBS-GRAIN-FEATURES). BBS-* segments aggregated at parent_route_id grain using DISTINCT hazard_id and broadcast to each segment; SPAN-* use span_id. Source: hazard_span_intersection_v4_candidate (is_within_radius). WARNING: NOT the v7 training source (v7 uses hazard_span_proximity); HSI counts run about 6x higher. Input for the Option B v8 challenger only. Candidate only.'
AS WITH sp AS (
  SELECT span_id, COALESCE(parent_route_id, span_id) AS route_key FROM {SPAN_V4}
),
h AS (
  SELECT COALESCE(sp.route_key, hs.span_id) AS route_key, hs.hazard_id, hs.hazard_type, hs.hazard_severity
  FROM {HSI_V4} hs
  LEFT JOIN sp ON hs.span_id = sp.span_id
  WHERE hs.is_within_radius
),
agg AS (
  SELECT route_key,
    COUNT(DISTINCT hazard_id) AS hazard_total_count,
    COUNT(DISTINCT CASE WHEN hazard_severity='critical' THEN hazard_id END) AS hazard_critical_count,
    COUNT(DISTINCT CASE WHEN hazard_type='wildfire' THEN hazard_id END) AS wildfire_count,
    COUNT(DISTINCT CASE WHEN hazard_type='flood' THEN hazard_id END) AS flood_count,
    COUNT(DISTINCT CASE WHEN hazard_type='severe_weather' THEN hazard_id END) AS weather_count,
    COUNT(DISTINCT CASE WHEN hazard_type='earthquake' THEN hazard_id END) AS earthquake_count,
    COUNT(DISTINCT CASE WHEN hazard_type='dig_ticket' THEN hazard_id END) AS dig_ticket_count,
    COUNT(*) AS in_radius_row_count
  FROM h GROUP BY route_key
)
SELECT sp.span_id, sp.route_key,
  CASE WHEN sp.route_key <> sp.span_id THEN 'parent_route' ELSE 'span' END AS feature_grain,
  CAST(COALESCE(a.hazard_total_count,0) AS DOUBLE) hazard_total_count,
  CAST(COALESCE(a.hazard_critical_count,0) AS DOUBLE) hazard_critical_count,
  CAST(COALESCE(a.wildfire_count,0) AS DOUBLE) wildfire_count,
  CAST(COALESCE(a.flood_count,0) AS DOUBLE) flood_count,
  CAST(COALESCE(a.weather_count,0) AS DOUBLE) weather_count,
  CAST(COALESCE(a.earthquake_count,0) AS DOUBLE) earthquake_count,
  CAST(COALESCE(a.dig_ticket_count,0) AS DOUBLE) dig_ticket_count,
  COALESCE(a.in_radius_row_count,0) AS in_radius_row_count
FROM sp LEFT JOIN agg a ON sp.route_key = a.route_key
"""

# Dry-run parity (read only): recomputed proximity rows vs the live candidate table.
prox_new = spark.sql(PROX_V4_SELECT)
prox_live = spark.table(OUT_PROX)
prox_diff = (prox_new.exceptAll(prox_live).count(), prox_live.exceptAll(prox_new).count())
print(f"{OUT_PROX}: recomputed={prox_new.count()} live={prox_live.count()} EXCEPT ALL new-live/live-new={prox_diff}")

if not UPSTREAM_WRITE:
    print("UPSTREAM_WRITE=false: upstream proximity table and HSI feature view not rebuilt.")
else:
    from ml_freeze_guard import assert_writable
    for target in (OUT_PROX, OUT_HSI_FEAT):
        assert target.endswith("_v4_candidate"), f"Refusing non-shadow target {target}"
        assert_writable(spark, target.split(".")[-1], lift_task=LIFT_TASK)
    spark.sql(PROX_V4_DDL)
    prox_new.createOrReplaceTempView("v4_prox_stage")
    # INSERT OVERWRITE keeps the table and column comments (semantic modeling mandate)
    spark.sql(f"INSERT OVERWRITE TABLE {SCHEMA}.hazard_span_proximity_v4_candidate SELECT * FROM v4_prox_stage")
    spark.sql(HSI_FEAT_VIEW_DDL)
    print(f"Wrote {spark.table(OUT_PROX).count()} rows to {OUT_PROX}; refreshed view {OUT_HSI_FEAT}")

In [0]:
def hazard_features(source_sql: str):
    """Proximity hazard pairs per span (<=10 km), by type. Same definitions as v7 training.

    weather_count matches hazard_type = 'weather', which is always 0. This is a known dead
    feature in v7 (the source label is 'severe_weather'). It is kept for parity and will be
    fixed in the v8 challenger (P47 finding).
    """
    return spark.sql(f"""
        SELECT span_id,
               count(*)                                      AS hazard_total_count,
               sum(int(hazard_type = 'wildfire'))            AS wildfire_count,
               sum(int(hazard_type = 'flood'))               AS flood_count,
               sum(int(hazard_type = 'dig_ticket'))          AS dig_ticket_count,
               sum(int(hazard_type = 'weather'))             AS weather_count,
               sum(int(hazard_type = 'earthquake'))          AS earthquake_count,
               sum(int(lower(hazard_severity) = 'critical')) AS hazard_critical_count
        FROM {source_sql}
        GROUP BY span_id""")

hz_v4 = hazard_features(PROX_V4)   # treatment: BBS-* segment geometry
hz_v3 = hazard_features(PROX_V3)   # control: v3 proximity at the parent BB-* route

log = spark.table(INFERENCE_LOG)
base_cols = [c for c in FEATURES if c not in HAZARD_FEATURES]

def feature_frame(hz, at_parent: bool):
    """v4 span grain (15,325 = 10,000 SPAN-* + 5,325 BBS-*). BBS-* segments inherit the 28
    non-hazard features from their parent BB-* route row in span_risk_inference_log."""
    spans = spark.table(SPAN_V4).select("span_id", F.coalesce("parent_route_id", "span_id").alias("route_key"))
    lg = log.select(F.col("span_id").alias("route_key"), *base_cols,
                    F.col("predicted_risk_class").alias("v3_prod_risk_class"))
    hz_key = "route_key" if at_parent else "span_id"
    return (spans.join(lg, "route_key", "inner")
                 .join(hz.withColumnRenamed("span_id", hz_key), hz_key, "left")
                 .fillna(0, subset=HAZARD_FEATURES)
                 .drop("route_key")
                 .toPandas())

pdf_v4 = feature_frame(hz_v4, at_parent=False)
pdf_v3 = feature_frame(hz_v3, at_parent=True)
assert len(pdf_v4) == 15325 and pdf_v4.span_id.is_unique, len(pdf_v4)
assert int(pdf_v4.hazard_total_count.max()) <= 20, "v7 out-of-distribution hazard counts"
print(len(pdf_v4), "spans |", int(pdf_v4.hazard_total_count.sum()), "v4 proximity pairs |", int(pdf_v3.hazard_total_count.sum()), "v3 proximity pairs (parent route)")

In [0]:
mlflow.set_registry_uri("databricks-uc")
mv = mlflow.MlflowClient().get_model_version_by_alias(MODEL_NAME, MODEL_ALIAS)
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{MODEL_ALIAS}")  # needs xgboost==3.4.1

def predict(pdf):
    out = model.predict(pdf[FEATURES].astype("float64"))
    return out if isinstance(out, pd.DataFrame) else pd.DataFrame(out)

pred_v4 = predict(pdf_v4)
pred_v3 = predict(pdf_v3)

out = pd.DataFrame({
    "span_id": pdf_v4.span_id.values,
    "predicted_risk_class": pred_v4["predicted_risk_class"].astype("int64").values,
    "predicted_risk_label": pred_v4["predicted_risk_label"].values,
    "prob_low": pred_v4["prob_low"].values,
    "prob_moderate": pred_v4["prob_moderate"].values,
    "prob_high": pred_v4["prob_high"].values,
    "prob_critical": pred_v4["prob_critical"].values,
    "hazard_total_count_v4": pdf_v4.hazard_total_count.astype("int64").values,
    "v3_prod_risk_class": pdf_v4.v3_prod_risk_class.astype("int64").values,
})
ctrl = pd.DataFrame({
    "span_id": pdf_v3.span_id.values,
    "v3rule_risk_class": pred_v3["predicted_risk_class"].astype("int64").values,
    "hazard_total_count_v3rule": pdf_v3.hazard_total_count.astype("int64").values,
})
out = out.merge(ctrl, on="span_id", how="left")

pred_sdf = (spark.createDataFrame(out)
    .withColumn("model_version", F.lit(f"uc_v{mv.version}_champion"))
    .withColumn("model_run_id", F.lit(mv.run_id))
    .withColumn("origin", F.lit("INFERENCE"))
    .withColumn("feature_source", F.lit(FEATURE_SOURCE))
    .withColumn("scored_at", F.current_timestamp())
    .select("span_id", "predicted_risk_class", "predicted_risk_label", "prob_low", "prob_moderate",
            "prob_high", "prob_critical", "v3rule_risk_class", "v3_prod_risk_class",
            "hazard_total_count_v4", "hazard_total_count_v3rule", "model_version", "model_run_id",
            "origin", "feature_source", "scored_at"))
pred_sdf.createOrReplaceTempView("v4_pred_stage")
print(f"champion v{mv.version} | critical spans v4={int((out.predicted_risk_class == 3).sum())} v3rule={int((out.v3rule_risk_class == 3).sum())}")

In [0]:
# Banker's rounding (bround) and an unrounded exposure inside overall_score reproduce the
# P34 pandas output (see the parity cell). Active hazard = in radius, started at or before
# SCORE_TIME, and not yet ended.
# P36-ML-V4-RESCORE-CONDUIT-DIVERSITY: path diversity = BUG-DP-070 conduit-grain score
# (CG-JACCARD-v1 = LEAST(v3, Jaccard over conduits + SRLG-ROUTE-*)). Falls back to v3 v22 if absent.
PD_V4 = f"{SCHEMA}.path_diversity_conduit_v4_candidate"

# BUG-QA-087 guard: HSI input must hold 0 retired BB-* span rows (v4 span grain is SPAN-* + BBS-*).
# A dry run only warns, so previews still work before the BUG-QA-073 rebuild. A write run hard-fails.
bb_rows = spark.sql(f"SELECT count(*) AS n FROM {HSI_V4} WHERE span_id LIKE 'BB-%' AND span_id NOT LIKE 'BBS-%'").first().n
if bb_rows:
    msg = f"BUG-QA-087: {HSI_V4} still has {bb_rows:,} retired BB-* rows (pre-BUG-QA-073). Snapshot hazards are not final."
    if WRITE_ENABLED:
        raise AssertionError(msg + " Refusing to write.")
    print("WARNING " + msg)
# P65-ML-SNAPSHOT-PIN-FP-GATE / BUG-QA-096: FROZEN v3 snapshot@22 through the fingerprint gate (fails closed).
V3_SNAP = v3src("resilience_score_snapshot", SNAPSHOT_V3_PIN)
snap_sdf = spark.sql(f"""
WITH hz AS (
  SELECT DISTINCT trim(sid) AS service_id, hazard_id
  FROM {HSI_V4}
  LATERAL VIEW explode(split(affected_service_ids, ',')) e AS sid
  WHERE is_within_radius
    AND hazard_start <= TIMESTAMP'{SCORE_TIME}'
    AND (hazard_end IS NULL OR hazard_end > TIMESTAMP'{SCORE_TIME}')
),
ah AS (SELECT service_id, count(DISTINCT hazard_id) AS active_hazards FROM hz GROUP BY service_id),
v3 AS (SELECT * FROM {V3_SNAP}),
b AS (
  SELECT v3.service_id, v3.score_time,
         -- DEC-PRI-NO-PROTECT-DIVERSITY-ZERO (BUG-QA-084): no protect path = 0 diversity, kept in averages
         CASE WHEN pd.has_protect_path = false THEN 0D
              ELSE coalesce(pd.path_diversity_score_v4, v3.path_diversity_score) END AS path_diversity_score, v3.telemetry_health_score,
         v3.srlg_count, v3.active_alarms, si.sla_tier, si.mrc_usd,
         coalesce(ah.active_hazards, 0) AS active_hazards,
         v3.active_hazards AS v3_active_hazards, v3.hazard_exposure_score AS v3_hazard_exposure_score,
         v3.overall_score AS v3_overall_score, v3.revenue_at_risk_usd AS v3_revenue_at_risk_usd
  FROM v3 JOIN {SERVICE_INVENTORY} si USING (service_id) LEFT JOIN ah USING (service_id)
  LEFT JOIN (SELECT service_id, has_protect_path, path_diversity_score_v4 FROM {PD_V4}) pd USING (service_id)
),
s AS (
  SELECT *,
    bround(100 * exp(-active_hazards * ln(2) / 20), 2) AS hazard_exposure_score,
    bround(least(100, 0.4 * path_diversity_score + 0.3 * (100 * exp(-active_hazards * ln(2) / 20))
                 + 0.3 * telemetry_health_score
                 + CASE sla_tier WHEN 'premium_plus' THEN 5 WHEN 'premium' THEN 2 ELSE 0 END), 2) AS overall_score
  FROM b
)
SELECT service_id, score_time, path_diversity_score, telemetry_health_score, srlg_count, active_alarms,
       sla_tier, mrc_usd, active_hazards, hazard_exposure_score, overall_score,
       bround(mrc_usd * (1 - overall_score / 100), 2) AS revenue_at_risk_usd,
       v3_active_hazards, v3_hazard_exposure_score, v3_overall_score, v3_revenue_at_risk_usd,
       CASE WHEN overall_score < 40 THEN 'critical' WHEN overall_score < 60 THEN 'high'
            WHEN overall_score < 80 THEN 'medium' WHEN overall_score < 90 THEN 'good' ELSE 'excellent' END AS risk_band,
       CASE WHEN v3_overall_score < 40 THEN 'critical' WHEN v3_overall_score < 60 THEN 'high'
            WHEN v3_overall_score < 80 THEN 'medium' WHEN v3_overall_score < 90 THEN 'good' ELSE 'excellent' END AS v3_risk_band,
       'DERIVED' AS origin, current_timestamp() AS created_at
FROM s""")
snap_sdf.createOrReplaceTempView("v4_snap_stage")
display(snap_sdf.agg(F.count("*").alias("services"), F.round(F.avg("overall_score"), 2).alias("avg_score"),
                     F.sum((F.col("overall_score") < 40).cast("int")).alias("critical"),
                     F.round(F.sum("revenue_at_risk_usd") / 1e6, 2).alias("revenue_at_risk_musd")))

In [0]:
# P50-ML-V4-BAND-RECALIBRATION / DEC-PRI-V4-BAND-CALIBRATION option (ii).
# BUG-QA-090: this cell must run AFTER the snapshot cell, which defines PD_V4 and v4_snap_stage.
# A separate v4 banding layer on top of the unchanged raw overall_score. It is display only:
# risk_band_v4 is NOT added to v4_snap_stage, so the guarded write cell is unaffected.
# The thresholds are PROPOSED and need CEO ratification at v4 cutover.
#
# Why the bands change: under DEC-PRI-NO-PROTECT-DIVERSITY-ZERO, a single-path service can score
# at most 0.3*100 + 0.3*100 + 5 = 65, so the v3 cuts (40/60/80/90) put 77-89% of services in critical.
# BAND-V4-ABS-v1 uses absolute cuts 15/30/45/60. They are fixed numbers, not quantiles, so they
# stay stable across runs. There is one structural rule: a service with no protect path is capped
# at 'medium', because a single path cannot be rated good or excellent for resilience.
BAND_V4_VERSION = "BAND-V4-ABS-v1"
BAND_V4_CUTS = (15, 30, 45, 60)  # critical < 15 <= high < 30 <= medium < 45 <= good < 60 <= excellent

band_sdf = spark.sql(f"""
WITH s AS (
  SELECT s.*, coalesce(pd.has_protect_path, true) AS has_protect_path
  FROM v4_snap_stage s LEFT JOIN (SELECT service_id, has_protect_path FROM {PD_V4}) pd USING (service_id)),
b AS (
  SELECT *, CASE WHEN overall_score < {BAND_V4_CUTS[0]} THEN 0 WHEN overall_score < {BAND_V4_CUTS[1]} THEN 1
                 WHEN overall_score < {BAND_V4_CUTS[2]} THEN 2 WHEN overall_score < {BAND_V4_CUTS[3]} THEN 3 ELSE 4 END AS raw_band_ix
  FROM s)
SELECT *, element_at(array('critical','high','medium','good','excellent'),
          1 + CASE WHEN NOT has_protect_path THEN least(raw_band_ix, 2) ELSE raw_band_ix END) AS risk_band_v4,
       '{BAND_V4_VERSION}' AS risk_band_v4_version
FROM b""")
band_sdf.createOrReplaceTempView("v4_band_preview")
display(spark.sql("""
  SELECT risk_band_v4, count(*) AS services, round(avg(overall_score), 2) AS avg_score,
         sum(int(NOT has_protect_path)) AS no_protect, round(sum(revenue_at_risk_usd) / 1e6, 2) AS rar_musd,
         sum(int(risk_band = 'critical')) AS critical_under_v3_cuts
  FROM v4_band_preview GROUP BY risk_band_v4
  ORDER BY array_position(array('critical','high','medium','good','excellent'), risk_band_v4)"""))

In [0]:
live_pred = spark.table(OUT_PRED).alias("l")
jp = pred_sdf.alias("n").join(live_pred, "span_id", "full")
pred_parity = jp.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("n.predicted_risk_class").eqNullSafe(F.col("l.predicted_risk_class")).cast("int")).alias("class_v4"),
    F.sum(F.col("n.v3rule_risk_class").eqNullSafe(F.col("l.v3rule_risk_class")).cast("int")).alias("class_v3rule"),
    F.sum(F.col("n.hazard_total_count_v4").eqNullSafe(F.col("l.hazard_total_count_v4")).cast("int")).alias("hz_v4"),
).first()

live_snap = spark.table(OUT_SNAP).alias("l")
js = snap_sdf.alias("n").join(live_snap, "service_id", "full")
snap_parity = js.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("n.active_hazards").eqNullSafe(F.col("l.active_hazards")).cast("int")).alias("active_hazards"),
    F.sum((F.abs(F.col("n.overall_score") - F.col("l.overall_score")) <= 0.01).cast("int")).alias("overall_within_0_01"),
    F.sum(F.col("n.risk_band").eqNullSafe(F.col("l.risk_band")).cast("int")).alias("risk_band"),
).first()
print("predictions:", pred_parity.asDict())
print("snapshot:   ", snap_parity.asDict())

In [0]:
if not WRITE_ENABLED:
    print("WRITE_ENABLED=false: dry run only, nothing written.")
else:
    from ml_freeze_guard import assert_writable
    for target in (OUT_PRED, OUT_SNAP):
        assert target.endswith("_v4_candidate"), f"Refusing non-shadow target {target}"
        assert_writable(spark, target.split(".")[-1], lift_task=LIFT_TASK)
    # INSERT OVERWRITE keeps the table and column comments (semantic modeling mandate)
    spark.sql(f"INSERT OVERWRITE TABLE {OUT_PRED} SELECT * FROM v4_pred_stage")
    spark.sql(f"INSERT OVERWRITE TABLE {OUT_SNAP} SELECT * FROM v4_snap_stage")
    spark.sql(f"""COMMENT ON COLUMN {OUT_SNAP}.path_diversity_score IS 'v4 path diversity 0-100. Conduit-grain CG-JACCARD-v1 from path_diversity_conduit_v4_candidate.path_diversity_score_v4 (BUG-DP-070). Services with no protect path score 0, not 100, and stay in fleet averages (DEC-PRI-NO-PROTECT-DIVERSITY-ZERO, BUG-QA-084).'""")
    print(f"Wrote {spark.table(OUT_PRED).count()} rows to {OUT_PRED}, {spark.table(OUT_SNAP).count()} rows to {OUT_SNAP}")